In [1]:
import pandas as pd

# 1. 팀원에게 받은 데이터 로드
new_research_file = 'semi_final_research_v3.csv'
df_new = pd.read_csv(new_research_file)

# 💡 [추가] 증폭 전, 원본 데이터 자체의 중복을 먼저 제거합니다.
# 텍스트(conversation)가 완전히 똑같은 데이터가 있다면 1개만 남깁니다.
print(f"🔍 중복 제거 전 데이터 개수: {len(df_new)}")
df_new = df_new.drop_duplicates(subset=['conversation']).reset_index(drop=True)
print(f"✨ 중복 제거 후 고유 데이터 개수: {len(df_new)}")

# 2. 조건별 분리 (is_edge 기준)
df_edge_1 = df_new[df_new['is_edge'] == 1].copy()
df_edge_0 = df_new[df_new['is_edge'] == 0].copy()

# 3. is_edge == 1인 데이터 15배 증폭
# 이제 중복이 없는 '깨끗한 100개(혹은 그 이하)'가 정확히 15배씩 복사됩니다.
df_edge_1_amplified = pd.concat([df_edge_1] * 15, ignore_index=True)

# 4. 증폭된 데이터와 나머지(is_edge=0) 합치기
df_combined_new = pd.concat([df_edge_1_amplified, df_edge_0], ignore_index=True)

# 5. 요청하신 컬럼 삭제 (is_edge, label 제거)
if 'is_edge' in df_combined_new.columns:
    df_combined_new.drop(['is_edge'], axis=1, inplace=True)
if 'label' in df_combined_new.columns:
    df_combined_new.drop(['label'], axis=1, inplace=True)

# 6. 합병용 파일로 저장
output_name = 'ready_to_merge_research.csv'
df_combined_new.to_csv(output_name, index=False)

print("="*40)
print(f"✅ 작업 완료! 파일명: {output_name}")
print(f"📊 최종 데이터 개수: {len(df_combined_new)}개")
print(f"🗑️ 삭제된 컬럼: is_edge, label")
print(f"📝 남은 컬럼: {list(df_combined_new.columns)}")
print("="*40)

🔍 중복 제거 전 데이터 개수: 295
✨ 중복 제거 후 고유 데이터 개수: 295
✅ 작업 완료! 파일명: ready_to_merge_research.csv
📊 최종 데이터 개수: 1695개
🗑️ 삭제된 컬럼: is_edge, label
📝 남은 컬럼: ['idx', 'conversation', 'class']


In [2]:
import pandas as pd

# 1. 기존 괴롭힘 데이터 로드 (파일 경로를 확인하세요)
# 만약 기존 데이터가 다른 파일에 있다면 그 파일을 로드해야 합니다.
origin_file = 'train_data.csv' 
df_origin = pd.read_csv(origin_file)

# 2. 합성 데이터(일반 대화) 로드
normal_file = 'ready_to_merge_research.csv'
df_normal = pd.read_csv(normal_file)

# 3. 데이터 합치기 (두 데이터를 위아래로 붙임)
train_data = pd.concat([df_origin, df_normal], axis=0).reset_index(drop=True)

print(f"합친 후 전체 데이터 개수: {len(train_data)}")
print("포함된 클래스 종류:", train_data['class'].unique())

# --- 이후에 이전에 작성한 전처리(label_dict 적용 등)를 수행하세요 ---
import pandas as pd
import re

print(f"초기 로드 데이터 개수: {len(train_data)}")

# 2. 표준 전처리 함수 정의
def standard_preprocess(text):
    if not isinstance(text, str):
        return ""
    # 공백 정규화
    text = re.sub(r'\s+', ' ', text)
    # URL 및 이메일 제거
    text = re.sub(r'http\S+|www\S+|mailto:\S+', '', text)
    # 한글, 숫자, 영문, 자음/모음, 기본 문장부호만 남기기
    text = re.sub(r'[^가-힣0-9a-zA-Zㄱ-ㅎㅏ-ㅣ\s.?!,]', '', text)
    return text.strip()

# 3. 전처리 적용 (변수명 train_data가 정확히 포함되어야 합니다)
train_data['conversation'] = train_data['conversation'].apply(standard_preprocess)

# 4. 클래스 라벨 맵핑 (모든 케이스 대응)
label_dict = {
    '협박 대화': 0, '협박': 0, '협박대화': 0,
    '갈취 대화': 1, '갈취': 1, '갈취대화': 1,
    '직장 내 괴롭힘 대화': 2, '직장내괴롭힘': 2, '직장내괴롭힘대화': 2,
    '기타 괴롭힘 대화': 3, '기타괴롭힘': 3, '기타괴롭힘대화': 3,
    '일반 대화': 4, '일반': 4, '일반대화': 4
}

# label 컬럼 생성
if 'class' in train_data.columns:
    train_data['label'] = train_data['class'].map(label_dict)
else:
    print("'class' 컬럼을 찾을 수 없습니다!")

# 5. 결측치 제거 및 데이터 셔플 (중복 제거 제외)
# 5-1. 라벨 결측치 제거
train_data = train_data.dropna(subset=['label'])

# 5-2. [핵심] 전체 데이터 셔플
# 데이터를 골고루 섞어서 특정 라벨이 몰려 있지 않게 합니다.
train_data = train_data.sample(frac=1, random_state=42).reset_index(drop=True)

# 6. 불필요한 컬럼 삭제 및 정리
if 'idx' in train_data.columns:
    train_data.drop(['idx'], axis=1, inplace=True)

final_df = train_data[['conversation', 'label']].copy()
print("-" * 30)
print(f"최종 정제 후 데이터 개수: {len(final_df)}")
print("\n[최종 라벨 분포]")
print(final_df['label'].value_counts().sort_index())
print("-" * 30)
print(final_df.head())

# 4. 전처리 후 최종 확인 (이 코드를 꼭 돌려서 '데이터 없음'이 안 나오는지 확인하세요)
for i in range(5):
    count = len(train_data[train_data['label'] == i])
    print(f"라벨 {i} 개수: {count}")

합친 후 전체 데이터 개수: 6540
포함된 클래스 종류: ['협박 대화' '기타 괴롭힘 대화' '갈취 대화' '직장 내 괴롭힘 대화' '일반 대화']
초기 로드 데이터 개수: 6540
------------------------------
최종 정제 후 데이터 개수: 6540

[최종 라벨 분포]
label
0     896
1     981
2     979
3    1094
4    2590
Name: count, dtype: int64
------------------------------
                                        conversation  label
0  돈 좀 줄 사람 얘 미안. 일단 가져와 안 돼 이 돈은. 뭐가 안돼 미안해 일단 쳐...      1
1  너 우리누나 좋아하지? 어? 어떻게 알았어? 니가 우리누나 지나갈때마다 음흉한 눈 ...      0
2  김대리! 스카프 샀어? 네네 너무 이쁘다 비싸보여 월급 받은 기념으로 하나 샀어요 ...      1
3  박과장님 여기 말씀하신 커피 타왔습니다. 어어 그래 김인턴 고마워. 역시 커피는 김...      2
4  존나 못생겼다. 너 유튜브에서 본 000이랑 졸라 똑같이 생겼다. 니 뒤통수 한 대...      3
라벨 0 개수: 896
라벨 1 개수: 981
라벨 2 개수: 979
라벨 3 개수: 1094
라벨 4 개수: 2590


In [8]:
import torch
import numpy as np
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    set_seed
)
import gc

# 1. 초기화 및 메모리 정리
gc.collect()
torch.cuda.empty_cache()

# 2. 공통 학습 세팅
MODEL_NAME = "klue/roberta-large"
MAX_LENGTH = 256
EPOCHS = 3
LEARNING_RATE = 2e-5  # 안정적인 학습을 위해 기존 5e-5에서 다시 조정
SEED = 42

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3. 데이터셋 준비 (final_df가 정의되어 있어야 합니다)
# 데이터를 무작위로 섞어주는 코드 추가
final_df = final_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# 그 다음에 기존 코드 실행
train_df, val_df = train_test_split(final_df, test_size=0.2, random_state=SEED, stratify=final_df['label'])

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["conversation"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.remove_columns(["conversation", "__index_level_0__"])
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_train.set_format("torch")

tokenized_val = tokenized_val.remove_columns(["conversation", "__index_level_0__"])
tokenized_val = tokenized_val.rename_column("label", "labels")
tokenized_val.set_format("torch")

# 4. 모델 로드 (최종 안정화 버전)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=5
).to(device)

# 체크포인팅 에러 방지를 위한 설정
model.config.use_cache = False 

# 5. TrainingArguments (가장 안정적인 수치를 내는 세팅)
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=1,      # 메모리 확보
    per_device_eval_batch_size=1,  
    gradient_accumulation_steps=16,     # 총 배치 사이즈
    
    fp16=False,                         # nan/unscale 에러 방지
    gradient_checkpointing=True,        # 메모리 절약 핵심
    
    eval_strategy="epoch",  
    save_strategy="epoch",
    logging_dir="./logs",
    seed=SEED,
    load_best_model_at_end=True,  
    metric_for_best_model="f1_macro",
    dataloader_pin_memory=False,
    optim="adamw_torch"
)

# 정확도 및 F1 계산 함수
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro')
    f1_weighted = f1_score(labels, preds, average='weighted')
    return {
        "accuracy": accuracy,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

# 6. 학습 시작
print("🚀 학습을 시작합니다...")
trainer.train()
print("✅ 학습 완료!")

# 7. 테스트 문장 및 예측 함수
id2label = {0: '협박 대화', 1: '갈취 대화', 2: '직장 내 괴롭힘 대화', 3: '기타 괴롭힘 대화', 4: '일반 대화'}

def predict_sentence_with_confidence(sentence):
    clean_sentence = standard_preprocess(sentence) # 전처리 함수 필수
    inputs = tokenizer(
        clean_sentence, 
        return_tensors="pt", 
        max_length=MAX_LENGTH, 
        padding="max_length", 
        truncation=True
    ).to(device)
    
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)
        max_prob, predicted_class_id = torch.max(probs, dim=-1)
        
    return id2label[predicted_class_id.item()], max_prob.item() * 100

# 8. 최종 결과 출력
print("\n" + "="*50)
print("📝 확신도가 포함된 테스트 문장 예측 결과")
print("="*50)

additional_test_sentences = [
    "너 어제 내가 말한 거 왜 처리 안 했어? 죄송해요 진짜 어제 몸이 너무 안 좋아서 그랬어요. 내 인내심 테스트하지 마. 오늘 밤까지 안 보내면 너네 집 주소 아는 사람 보낼 거야. 진짜 죽기 싫으면 제대로 해라.",
    "야 이번에 부모님한테 용돈 받았다며? 응 근데 이건 진짜 학원비 내야 하는 돈이라서 안 돼. 야 학원비는 나중에 내고 일단 좀 내놔봐. 좋은 말로 할 때 주는 게 서로 편하지 않겠냐? 내일까지 이자 붙여서 가져와.",
    "이대리 이 기획안 제정신으로 쓴 거야? 초등학생도 이것보다는 잘 쓰겠다. 죄송합니다 다시 수정해서 보고드리겠습니다. 다시 한다고 뭐가 달라져? 너 같은 무능한 애가 우리 팀에 있다는 게 진짜 민폐다 그냥 오늘 당장 사표 쓰고 나가.",
    "야 쟤 또 저 옷 입고 왔네? 진짜 냄새나게 생겼다. 야 들리겠어 그만해. 들으라고 하는 소리야 너는 거울도 안 보고 사냐? 우리 쪽으로 오지 마 재수 없으니까.",
    "팀장님 오늘 회의록 정리해서 메일로 보내드렸습니다. 아 방금 확인했습니다 고생 많으셨어요. 별말씀을요 추가로 수정할 부분 있으시면 퇴근 전에 말씀해 주세요!"
]

for text in additional_test_sentences:
    pred_label, confidence = predict_sentence_with_confidence(text)
    print(f"문장: {text}")
    print(f"결과: {pred_label} (확신도: {confidence:.2f}%)\n")

Map:   0%|          | 0/5232 [00:00<?, ? examples/s]

Map:   0%|          | 0/1308 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


🚀 학습을 시작합니다...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,No log,0.348500,0.922783,0.902436,0.922817
2,4.528380,0.319994,0.946483,0.930379,0.946787
3,4.528380,0.311165,0.953364,0.938603,0.953457


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ 학습 완료!

📝 확신도가 포함된 테스트 문장 예측 결과
문장: 너 어제 내가 말한 거 왜 처리 안 했어? 죄송해요 진짜 어제 몸이 너무 안 좋아서 그랬어요. 내 인내심 테스트하지 마. 오늘 밤까지 안 보내면 너네 집 주소 아는 사람 보낼 거야. 진짜 죽기 싫으면 제대로 해라.
결과: 협박 대화 (확신도: 99.98%)

문장: 야 이번에 부모님한테 용돈 받았다며? 응 근데 이건 진짜 학원비 내야 하는 돈이라서 안 돼. 야 학원비는 나중에 내고 일단 좀 내놔봐. 좋은 말로 할 때 주는 게 서로 편하지 않겠냐? 내일까지 이자 붙여서 가져와.
결과: 일반 대화 (확신도: 99.99%)

문장: 이대리 이 기획안 제정신으로 쓴 거야? 초등학생도 이것보다는 잘 쓰겠다. 죄송합니다 다시 수정해서 보고드리겠습니다. 다시 한다고 뭐가 달라져? 너 같은 무능한 애가 우리 팀에 있다는 게 진짜 민폐다 그냥 오늘 당장 사표 쓰고 나가.
결과: 직장 내 괴롭힘 대화 (확신도: 99.99%)

문장: 야 쟤 또 저 옷 입고 왔네? 진짜 냄새나게 생겼다. 야 들리겠어 그만해. 들으라고 하는 소리야 너는 거울도 안 보고 사냐? 우리 쪽으로 오지 마 재수 없으니까.
결과: 기타 괴롭힘 대화 (확신도: 99.99%)

문장: 팀장님 오늘 회의록 정리해서 메일로 보내드렸습니다. 아 방금 확인했습니다 고생 많으셨어요. 별말씀을요 추가로 수정할 부분 있으시면 퇴근 전에 말씀해 주세요!
결과: 일반 대화 (확신도: 100.00%)



In [9]:
import pandas as pd
import torch
from tqdm import tqdm

# 1. 데이터 로드 (파일 경로를 확인해 주세요)
# 사용자님이 말씀하신 idx, conversation 컬럼이 있는 파일입니다.
test_df = pd.read_csv('./data/test.csv') 

# 2. 추론 모드 설정
model.eval()
submission_list = []

print("🚀 Kaggle 제출용 추론 시작 (컬럼명: idx, target)...")

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    # test.csv의 컬럼명 사용
    idx = row['idx']
    text = row['conversation']
    
    # 학습 시와 동일한 전처리 적용 (필수)
    if 'standard_preprocess' in globals():
        text = standard_preprocess(text)
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        # 숫자로 된 클래스 번호(0, 1, 2, 3, 4)를 바로 가져옵니다.
        predicted_class_id = torch.argmax(outputs.logits, dim=-1).item()

    # 사이트 안내에 맞게 idx와 target 컬럼 구성
    submission_list.append({
        'idx': idx,
        'target': predicted_class_id
    })

# 3. submission.csv 저장
submission_df = pd.DataFrame(submission_list)
submission_df.to_csv('submission.csv', index=False)

print("\n✅ submission.csv 생성 완료!")
print(submission_df.head()) # 상위 5개 행을 출력해 형식을 확인합니다.

🚀 Kaggle 제출용 추론 시작 (컬럼명: idx, target)...


100%|██████████| 500/500 [00:33<00:00, 14.71it/s]


✅ submission.csv 생성 완료!
     idx  target
0  t_000       1
1  t_001       2
2  t_002       2
3  t_003       4
4  t_004       3


In [10]:
file_name = 'final_data.csv'
final_df.to_csv(file_name, index=False, encoding='utf-8-sig')

print(f"✅ {file_name} 파일 저장 완료!")

✅ final_data.csv 파일 저장 완료!


In [5]:
!du -sh * | sort -hr

58G	results
2.7M	data
2.6M	train_data.csv
2.4M	final_train_data.csv
420K	final_normal_conversations_1000.csv
364K	ready_to_merge_research.csv
132K	DATA_EDA.ipynb
60K	semi_final_research_v3.csv
24K	FINAL_DATA.ipynb
16K	submission_class.csv
4.0K	Untitled.ipynb
4.0K	README.md


In [7]:
!rm -rf ./results/*